# Agent Loop：先看观察怎样改变下一步

本实验调用[完整Python源码](../05-code/agent-loop-python/README.md)，数据是三条人工编写文档，策略是规则，不是LLM。先预测每种情况的停止原因，再执行对照。机制见[Loop文章](../01-concepts/02-agent-loop.md)。

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "10-Knowledge").is_dir())
sys.path.insert(0, str(ROOT / "10-Knowledge/03-agent-core/05-code/agent-loop-python/src"))
from agent_loop import Action, EvidenceModel, ScriptedModel, default_tools, run_agent
from dataclasses import asdict


## 正常完成：搜索一次，再依据结果回答

打印每一次工具观察。答案中的文档ID来自返回值；这只验证来源ID传递，不证明任意自然语言主张都被证据支持。

In [2]:
state = run_agent(EvidenceModel(), default_tools(), "上下文")
print("决策次数:", state.steps, "状态:", state.status, "原因:", state.stop_reason)
print(json.dumps(state.observations, ensure_ascii=False, indent=2))
print(state.answer)
assert state.steps == 2 and "[doc-1]" in state.answer

决策次数: 2 状态: completed 原因: model_finished
[
  {
    "call_id": "48aa021f01234e038bdddd6007e9e8ef:1",
    "ok": true,
    "data": {
      "documents": [
        {
          "id": "doc-1",
          "text": "上下文预算必须为模型输出和工具结果预留空间。"
        },
        {
          "id": "doc-2",
          "text": "上下文压缩应保留当前目标、约束、证据来源和未完成任务。"
        }
      ]
    }
  }
]
[doc-1] 上下文预算必须为模型输出和工具结果预留空间。
[doc-2] 上下文压缩应保留当前目标、约束、证据来源和未完成任务。


## 同样搜索没有新信息时，程序停止

动作、参数和结果相同才算重复；本实现两次相同结果后停止。这个阈值用于教学，不能套给需要重复采样的任务。

In [3]:
actions = [Action("tool", "search", {"query":"上下文"})] * 5
repeat = run_agent(ScriptedModel(actions), default_tools(), "上下文")
budget = run_agent(ScriptedModel(actions), default_tools(), "上下文", max_steps=1)
print([(s.stop_reason, s.steps) for s in [state, repeat, budget]])
assert repeat.stop_reason == "no_progress"
assert budget.stop_reason == "max_steps"

[('model_finished', 2), ('no_progress', 2), ('max_steps', 1)]


## 错参数不会调用成功，错误作为观察保留

数字7虽然能写进JSON，但搜索函数要求字符串。下一轮策略可以修参数，权限与参数规则本身不因此放宽。

In [4]:
bad = run_agent(ScriptedModel([Action("tool", "search", {"query":7}), Action("finish", answer="参数错误，结束此次示例。")]), default_tools(), "x")
print(bad.observations[0])
assert bad.observations[0]["error"]["code"] == "invalid_arguments"

{'call_id': '3e492872d64044b680a04a9abbf7e3af:1', 'ok': False, 'error': {'code': 'invalid_arguments', 'retryable': False}}


## 读取实际Trace

本实现结束时把事件写入新JSONL文件。它用于诊断，不能在进程中途崩溃后恢复。

In [5]:
import tempfile
with tempfile.TemporaryDirectory() as folder:
    trace_path = str(Path(folder) / "trace.jsonl")
    run_agent(EvidenceModel(), default_tools(), "预算", trace_path=trace_path)
    events = [json.loads(line) for line in Path(trace_path).read_text(encoding="utf-8").splitlines()]
    print([(e["seq"], e["kind"]) for e in events])
    assert events[-1]["kind"] == "run_finished"

[(0, 'model_decision'), (1, 'tool_call'), (2, 'tool_result'), (3, 'model_decision'), (4, 'run_finished')]


## 你应能解释的结果

正常任务需要2次决策，重复动作在第2次后停止，预算1只允许一次搜索。把重复阈值改成3、把查询改成不存在的词，先预测结果再运行。无命中是成功执行得到空结果，与工具报错不同。本实验不测真实模型动作选择；后续可对照 [workbench 已保存的本地模型结果](../../../20-Projects/learning-workbench/artifacts/real-models/agent-comparison.json)，继续区分选对工具、正常完成与答案匹配。